In [13]:
import boto3
import pandas as pd
import io

athena = boto3.client("athena", region_name="us-east-1")

def run_query(sql, db="aqi_db"):
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": db, "Catalog": "AwsDataCatalog"},
        ResultConfiguration={"OutputLocation": "s3://weather-bulk/athena-results/"},
        WorkGroup="primary",
    )
    eid = resp["QueryExecutionId"]
    import time
    while True:
        state = athena.get_query_execution(QueryExecutionId=eid)["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Query {state}")
        time.sleep(2)
    pages = athena.get_query_results(QueryExecutionId=eid)
    cols = [c["Label"] for c in pages["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]]
    rows = [[c.get("VarCharValue", "") for c in r["Data"]] for r in pages["ResultSet"]["Rows"][1:]]
    return pd.DataFrame(rows, columns=cols)


In [14]:
sql = """
SELECT COUNT(*) AS row_count, MAX(timestamp) AS latest, MIN(timestamp) AS earliest
FROM aqi_db.aqi_unified
WHERE source = 'hourly'
  AND timestamp >= (current_timestamp - INTERVAL '1' HOUR)
"""

df = run_query(sql)
print(df.to_string(index=False))

if int(df["row_count"].iloc[0]) > 0:
    print(f"\n✅ Data ingested in the last hour — {df['row_count'].iloc[0]} rows, latest: {df['latest'].iloc[0]}")
else:
    print("\n❌ No data ingested in the last hour — Lambda may not have run yet")


row_count latest earliest
        0                

❌ No data ingested in the last hour — Lambda may not have run yet


In [15]:
import boto3
from datetime import datetime, timezone

s3 = boto3.client("s3", region_name="us-east-1")

# Check S3 for hourly files written in last 2 hours
now = datetime.now(timezone.utc)
prefix = f"hourly/year={now.year}/month={now.month:02d}/day={now.day:02d}/"

resp = s3.list_objects_v2(Bucket="weather-bulk", Prefix=prefix)
files = resp.get("Contents", [])

if files:
    print(f"✅ Found {len(files)} hourly file(s) in S3 today under {prefix}")
    for f in files:
        print(f"  {f['Key']}  ({f['LastModified'].strftime('%H:%M UTC')})")
else:
    print(f"❌ No files found in s3://weather-bulk/{prefix}")
    print("   → Lambda A has not run yet today, or EventBridge is not firing")


✅ Found 4 hourly file(s) in S3 today under hourly/year=2026/month=03/day=20/
  hourly/year=2026/month=03/day=20/aqi_2026-03-20_20.parquet  (20:24 UTC)
  hourly/year=2026/month=03/day=20/aqi_2026-03-20_21.parquet  (21:24 UTC)
  hourly/year=2026/month=03/day=20/aqi_2026-03-20_22.parquet  (22:24 UTC)
  hourly/year=2026/month=03/day=20/aqi_2026-03-20_23.parquet  (23:24 UTC)


In [16]:
# Fix: repair raw_hourly partitions so Athena sees new S3 files, then re-check
run_query("MSCK REPAIR TABLE aqi_db.raw_hourly")
print("✅ Partitions repaired")

df2 = run_query("""
SELECT COUNT(*) AS row_count, MAX(timestamp) AS latest
FROM aqi_db.aqi_unified
WHERE source = 'hourly'
  AND timestamp >= (current_timestamp - INTERVAL '2' HOUR)
""")
print(df2.to_string(index=False))


✅ Partitions repaired
row_count latest
        0       
